In [4]:
# Install required libraries (only needed in Colab)
!pip install tensorflow opencv-python

# Import libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import json

# Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATASET_PATH = "/content/drive/MyDrive/data_photo"

# Data Preparation
# ImageDataGenerator handles preprocessing and validation split
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Load training data
train = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

# Load validation data
val = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

# Save class names (folder names = labels)
class_names = list(train.class_indices.keys())

# Model Creation
# Load pretrained MobileNetV2 (without top classification layer)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze base model (transfer learning)
base_model.trainable = False

# Create custom classification head
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

# Compile model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


# Training
model.fit(
    train,
    validation_data=val,
    epochs=5
)

# Saving Model
# Save trained model
model.save("dog_model.h5")

# Save class labels
with open("classes.json", "w") as f:
    json.dump(class_names, f)

Found 2375 images belonging to 30 classes.
Found 593 images belonging to 30 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 876s 12s/step - accuracy: 0.4720 - loss: 1.9435 - val_accuracy: 0.6071 - val_loss: 1.3554
Epoch 2/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 166s 2s/step - accuracy: 0.7596 - loss: 0.8158 - val_accuracy: 0.6408 - val_loss: 1.2863
Epoch 3/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 176s 2s/step - accuracy: 0.8375 - loss: 0.5680 - val_accuracy: 0.6695 - val_loss: 1.2533
Epoch 4/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 138s 2s/step - accuracy: 0.8728 - loss: 0.4336 - val_accuracy: 0.6661 - val_loss: 1.1942
Epoch 5/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 139s 2s/step - accuracy: 0.9048 - loss: 0.3166 - val_accuracy: 0.6914 - val_loss: 1.2046
